# 06 — Target eligibility

This notebook validates the frozen 15-minute MES target definition in memory. It does not write target data, create features, or evaluate predictive performance.

Each observed one-minute decision boundary $T$ uses the first V2 event-time trade in $[T, T+10s)$ as the start reference and the corresponding first trade in $[T+15m, T+15m+10s)$ as the future reference. The elapsed minute path must remain continuous, in one CME session, and in one contract. Equal endpoint prices are excluded from the binary target.

## Scope

The target price is outcome data only. It must never be added to a feature vector at $T$. The checks below exercise the implementation on representative sessions before any full-history target layer is built.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.target_engineering_v1 import (
    construct_15m_target_for_session,
    load_one_minute_decision_rows,
    load_v2_one_second_session,
    run_deterministic_target_tests,
    summarize_target_session,
)

## Deterministic boundary checks

These small in-memory checks lock the strict 10-second rule, target direction, internal-minute continuity, session and roll protection, early-close behavior, and source-boundary handling.

In [ ]:
pd.Series(run_deterministic_target_tests(), name='result')

## Representative-session validation

The selection covers an ordinary session, a scheduled early close, the November outage, unresolved December gaps, a roll, and both source-boundary partial sessions. Counts are construction diagnostics, not model results.

In [ ]:
REPRESENTATIVE_SESSIONS = [
    '2026-09-08',  # ordinary full session
    '2025-11-27',  # scheduled early close
    '2025-11-28',  # confirmed CME outage
    '2025-12-24',  # unresolved isolated gaps
    '2025-12-30',  # unresolved isolated gap
    '2025-12-17',  # contract-roll session
    '2025-10-07',  # first source-boundary partial session
    '2026-09-11',  # final source-boundary partial session
]

session_targets = {}
session_summaries = {}
for session_date in REPRESENTATIVE_SESSIONS:
    minute_rows = load_one_minute_decision_rows(session_date)
    seconds = load_v2_one_second_session(session_date)
    target_rows = construct_15m_target_for_session(minute_rows, seconds)
    session_targets[session_date] = target_rows
    session_summaries[session_date] = summarize_target_session(target_rows)

pd.DataFrame(session_summaries).T.drop(columns='ineligible_by_reason')

In [ ]:
pd.DataFrame.from_dict(
    {
        session_date: summary['ineligible_by_reason']
        for session_date, summary in session_summaries.items()
    },
    orient='index',
).fillna(0).astype(int)

## Full-history Target V1 build

Target V1 has been built at `data/processed/targets/target_v1/`. The source module wrote one session at a time, read every file back, and reconciled it to the matching frozen one-minute and V2 source sessions. No features or models were created.

In [ ]:
import json

TARGET_DIR = ROOT / 'data' / 'processed' / 'targets' / 'target_v1'
with (TARGET_DIR / 'run_manifest.json').open() as handle:
    target_manifest = json.load(handle)

pd.Series({
    'session_files': target_manifest['session_files'],
    'candidate_rows': target_manifest['total_candidate_rows'],
    'structurally_valid_paths': target_manifest['structurally_valid_paths'],
    'reference_pairs_within_10s': target_manifest['reference_pairs_within_10s'],
    'equal_price_count': target_manifest['equal_price_count'],
    'binary_eligible_rows': target_manifest['binary_eligible_rows'],
}, name='Target V1 reconciliation')

In [ ]:
audit = pd.DataFrame(target_manifest['session_audit']).set_index('session_date')
audit.loc[[
    '2025-10-07', '2025-11-27', '2025-11-28', '2025-12-17',
    '2025-12-24', '2025-12-30', '2026-09-11',
], ['candidate_rows', 'structurally_valid_paths', 'reference_pairs_within_10s',
    'equal_price_count', 'binary_eligible_rows', 'up_count', 'down_count']]